In [ ]:
# [fugu-core]
import os
from dataclasses import dataclass
from typing import Any
from pydantic import BaseModel, Field

PROVIDER, MODEL = os.getenv("VIDBYTE_COOKBOOK_PROVIDER", "openai"), os.getenv("VIDBYTE_COOKBOOK_MODEL", "gpt-4.1")
PROVIDER_ENV = {"openai": "OPENAI_API_KEY", "anthropic": "ANTHROPIC_API_KEY", "gemini": "GEMINI_API_KEY", "xai": "XAI_API_KEY", "deepseek": "DEEPSEEK_API_KEY"}
MAX_RECURSION_DEPTH, TOKEN_BUDGET, MAX_COST, COST_PER_MTOK, MAX_CALLS, MAX_TOOL_ROUNDS = 2, 20_000, 0.30, 4.0, 12, 5

ORCHESTRATOR_PROMPT = """You are Fugu, an orchestration model. Return a typed plan, not an answer. Use mode='direct' for simple single-pass asks. Use mode='delegate' for hard, multi-step, or multi-domain asks; create independent subtasks and assign each to one key: reasoning, coding, long_context, fast, or fugu."""
SYNTHESIZER_PROMPT = """Combine the original request and labeled expert outputs into one complete answer. Reconcile conflicts, remove redundancy, and do not mention the orchestration machinery."""
VERIFIER_PROMPT = """Check the draft against the original request, fix errors or gaps, and return only the final answer."""
EXPERT_PROMPT = """You are a {specialty} expert inside Fugu. Solve the assigned subtask completely. Use search/read_article for factual grounding when useful."""

class Subtask(BaseModel):
    description: str = Field(description="Self-contained work for one expert.")
    assigned_to: str = Field(description="One of: reasoning, coding, long_context, fast, fugu.")

class OrchestrationPlan(BaseModel):
    mode: str = Field(description="direct or delegate")
    reasoning: str = Field(description="Why this route was chosen.")
    subtasks: list[Subtask] = Field(default_factory=list)

@dataclass
class PoolModel:
    key: str; provider: str; model: str; agent: Any

class Fugu:
    def __init__(self, orchestrator, pool, synthesizer, verifier=None, *, ultra=False, exclude=(), verbose=False):
        self.orchestrator, self.pool, self.synthesizer, self.verifier = orchestrator, list(pool), synthesizer, verifier
        self.by_key, self.ultra, self.exclude, self.verbose = {m.key: m for m in self.pool}, ultra, set(exclude), verbose
    def run(self, prompt, _depth=0):
        plan = self._orchestrate(prompt); self._say(f"route={plan.mode} ({plan.reasoning})")
        if plan.mode != "delegate" or not plan.subtasks: return self._solve_directly(prompt)
        answer = self._synthesize(prompt, [self._run_subtask(t, _depth) for t in plan.subtasks])
        return self._verify(prompt, answer) if self.ultra and self.verifier else answer
    def _orchestrate(self, prompt):
        try:
            payload = self.orchestrator.run(prompt).metadata.get("structured")
            return payload if isinstance(payload, OrchestrationPlan) else OrchestrationPlan.model_validate(payload)
        except Exception:
            return OrchestrationPlan(mode="direct", reasoning="fallback: no structured plan")
    def _available(self, key): return key in self.by_key and key not in self.exclude and bool(os.getenv(PROVIDER_ENV.get(self.by_key[key].provider, "")))
    def _resolve(self, key, _skip=()):
        if key not in _skip and self._available(key): return self.by_key[key]
        for member in self.pool:
            if member.key not in _skip and self._available(member.key):
                if key != "__any__" and member.key != key: self._say(f"routing around '{key}' -> '{member.key}'")
                return member
        raise RuntimeError("Fugu: no available models in pool")
    def _run_subtask(self, task, depth):
        if task.assigned_to == "fugu" and depth < MAX_RECURSION_DEPTH:
            self._say(f"recursing into Fugu depth {depth + 1}"); return f"[fugu] {self.run(task.description, _depth=depth + 1)}"
        member = self._resolve(task.assigned_to)
        try: return f"[{member.key}] {member.agent.run(task.description).content}"
        except Exception as exc:
            self._say(f"'{member.key}' failed ({exc}); routing around")
            substitute = self._resolve(member.key, _skip={member.key})
            return f"[{substitute.key}] {substitute.agent.run(task.description).content}"
    def _dispatch(self, subtasks, depth): return [self._run_subtask(t, depth) for t in subtasks]
    def _synthesize(self, prompt, outputs): return self.synthesizer.run(f"Original request:\n{prompt}\n\nExpert outputs:\n" + "\n\n".join(outputs)).content
    def _verify(self, prompt, answer): self._say("ultra: verification pass"); return self.verifier.run(f"Request:\n{prompt}\n\nDraft answer:\n{answer}").content
    def _solve_directly(self, prompt): return self._resolve("__any__").agent.run(prompt).content
    def _say(self, msg):
        if self.verbose: print(f"  [fugu] {msg}")

def build_fugu(*, ultra=False, exclude=(), verbose=True):
    import re, requests
    from vidbyte import BaseAgent, tool
    from vidbyte.middleware import TokenBudgetMiddleware, CostBudgetMiddleware, RuntimeLimitMiddleware, ModelRetryMiddleware
    @tool
    def search(query: str) -> list[dict]:
        r = requests.get("https://en.wikipedia.org/w/api.php", headers={"User-Agent": "vidbyte-cookbook-sakana-fugu/0.1"}, timeout=30, params={"action": "query", "list": "search", "srsearch": query, "srlimit": 5, "format": "json"}); r.raise_for_status()
        return [{"title": h["title"], "snippet": re.sub(r"<[^>]+>", "", h["snippet"])} for h in r.json()["query"]["search"]]
    @tool
    def read_article(title: str) -> str:
        r = requests.get("https://en.wikipedia.org/w/api.php", headers={"User-Agent": "vidbyte-cookbook-sakana-fugu/0.1"}, timeout=30, params={"action": "query", "prop": "extracts", "explaintext": 1, "titles": title, "format": "json", "redirects": 1}); r.raise_for_status()
        return (next(iter(r.json()["query"]["pages"].values())).get("extract") or f"No article found for {title!r}.")[:6000]
    def guards(): return [TokenBudgetMiddleware(max_tokens=TOKEN_BUDGET), CostBudgetMiddleware(max_spend_usd=MAX_COST, cost_per_million_tokens=COST_PER_MTOK), RuntimeLimitMiddleware(max_model_calls=MAX_CALLS, max_elapsed_seconds=120), ModelRetryMiddleware(max_attempts=2)]
    def agent(name, prompt, **kw): return BaseAgent(name=name, system_prompt=prompt, provider=PROVIDER, model_name=MODEL, middleware=guards(), **kw)
    orchestrator = agent("fugu-orchestrator", ORCHESTRATOR_PROMPT, output_schema=OrchestrationPlan)
    synthesizer, verifier = agent("fugu-synthesizer", SYNTHESIZER_PROMPT), agent("fugu-verifier", VERIFIER_PROMPT)
    def expert(key, provider, model, specialty): return PoolModel(key, provider, model, BaseAgent(name=f"expert-{key}", system_prompt=EXPERT_PROMPT.format(specialty=specialty), provider=provider, model_name=model, tools=[search, read_article], middleware=guards(), max_tool_rounds=MAX_TOOL_ROUNDS))
    pool = [expert("reasoning", "anthropic", "claude-sonnet-4-6", "deep reasoning"), expert("coding", "openai", MODEL, "software engineering"), expert("long_context", "gemini", "gemini-2.5-pro", "long-context synthesis"), expert("fast", "deepseek", "deepseek-chat", "fast drafting")]
    return Fugu(orchestrator, pool, synthesizer, verifier=verifier, ultra=ultra, exclude=exclude, verbose=verbose)

# Usage: fugu = build_fugu(); fugu.run("Compare why Concorde retired with what a modern supersonic revival must solve.")
